### Load and Inspect

In [1]:
# Load libs
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
import papermill as pm
from datetime import datetime, timedelta
from tqdm import tqdm

# Load data
DB_PATH = 'data/data.db'

with sqlite3.connect(DB_PATH) as conn:
    raw = pd.read_sql("SELECT * FROM ohlc_hw", conn)

# simple inspection
raw.head()

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [2]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


In [3]:
raw.describe()

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


In [4]:
# display date range 
print("Date range: ", raw['ts'].min(), "-", raw['ts'].max())
# display number and list of unique tickers
print("Number of unique tickers: ", raw['ticker'].nunique())
sorted(raw['ticker'].unique())

Date range:  1962-01-02 - 2024-12-06
Number of unique tickers:  150


['<a target="_blank"><span style="color: blue">AAPL</span></a>',
 '<a target="_blank"><span style="color: blue">AMGN</span></a>',
 '<a target="_blank"><span style="color: blue">AMZN</span></a>',
 '<a target="_blank"><span style="color: blue">AXP</span></a>',
 '<a target="_blank"><span style="color: blue">BA</span></a>',
 '<a target="_blank"><span style="color: blue">CAT</span></a>',
 '<a target="_blank"><span style="color: blue">CRM</span></a>',
 '<a target="_blank"><span style="color: blue">CSCO</span></a>',
 '<a target="_blank"><span style="color: blue">CVX</span></a>',
 '<a target="_blank"><span style="color: blue">DIS</span></a>',
 '<a target="_blank"><span style="color: blue">GS</span></a>',
 '<a target="_blank"><span style="color: blue">HD</span></a>',
 '<a target="_blank"><span style="color: blue">HON</span></a>',
 '<a target="_blank"><span style="color: blue">IBM</span></a>',
 '<a target="_blank"><span style="color: blue">JNJ</span></a>',
 '<a target="_blank"><span style="color

In [5]:
# normalize ticker labels for correct ticker counts
df = raw.copy()
df['ticker_raw'] = df['ticker']

# clean up the html labeling for each ticker
df['ticker'] = (
    df['ticker']
    .str.replace(r"<[^>]+>", "", regex = True)  # strip the html tag
    .str.strip()                                # remove white space
    .str.upper()
)

print('Unique ticker before: ', raw['ticker'].nunique())
print('Unique ticker after: ', df['ticker'].nunique())
# Correct 30 distinct tickers
sorted(df['ticker'].unique())

Unique ticker before:  150
Unique ticker after:  30


['AAPL',
 'AMGN',
 'AMZN',
 'AXP',
 'BA',
 'CAT',
 'CRM',
 'CSCO',
 'CVX',
 'DIS',
 'GS',
 'HD',
 'HON',
 'IBM',
 'JNJ',
 'JPM',
 'KO',
 'MCD',
 'MMM',
 'MRK',
 'MSFT',
 'NKE',
 'NVDA',
 'PG',
 'SHW',
 'TRV',
 'UNH',
 'V',
 'VZ',
 'WMT']

Inspect summary data again

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   ts          397711 non-null  object 
 1   open        397711 non-null  float64
 2   high        397711 non-null  float64
 3   low         397711 non-null  float64
 4   close       397711 non-null  float64
 5   adj_close   377856 non-null  float64
 6   volume      397711 non-null  int64  
 7   ticker      397711 non-null  object 
 8   ticker_raw  397711 non-null  object 
dtypes: float64(5), int64(1), object(3)
memory usage: 27.3+ MB


In [7]:
df.describe()

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


There may exist duplicates. Check duplicates and remove them

In [8]:
# check exact duplicates
cols = ['ticker', 'ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
exact_dups = df.duplicated(subset = cols, keep = 'first')
print('Exact duplicates: ', exact_dups.sum())

# check dups of same (ticker, ts) but different values 
dedup = df[~exact_dups]
key_counts = dedup.groupby(['ticker', 'ts']).size()
print("conflicting (ticker, ts) keys: ", (key_counts > 1).sum())

Exact duplicates:  29460
conflicting (ticker, ts) keys:  0


Next remove the exact duplicates from the working df and proceed to the next steps

In [9]:
duplicate_rows = df[exact_dups].copy()
df = df[~exact_dups].copy()
print("Rows after removing exact duplicates: ", len(df))

Rows after removing exact duplicates:  368251


**Columns Description**
- `ts`: trading date
- `ticker`: stock symbol
- `open`, `high`, `low`, `close`: raw daily prices (first, highest, lowest, last trade)
- `adj_close`: close adjusted for splits and dividends
- `volume`: number of shares traded

### 1.2 Detect Data Quality Issues

First build a helper function for final data quality reports

In [10]:
REPORT_COLS = ['ticker', 'issue_type', 'issue_date', 'description', 'severity']

# turn the flagged rows into data quality report format
def make_issues(rows, issue_type, description, severity):
    if isinstance(description, pd.Series): description = description.values
    out = pd.DataFrame({
        'ticker': rows['ticker'].values,
        'issue_type': issue_type,
        'issue_date': rows['ts'].values,
        'description': description,
        'severity': severity
    })
    return out[REPORT_COLS]

**Severity Definitions**

Here I define each serverity level as the following:

- **High**: the value is impossible or unusable in the table and must be repaired or quarantined
  (nonpositive/non-finite prices, OHLC bound violations, missing raw prices, conflicting keys,
  `adj_close` inconsistent with the reference adjustment factor).

- **Medium**: the value is probably wrong or incomplete, with a documented repair available
  (missing `adj_close`, missing trading days, exact duplicate records).
  
- **Low**: the value is possibly correct but flagged for review only and not deleted from the table
  (3σ return outliers, zero-volume days).

Next we handle missing values

In [11]:
def detect_missing_values(df):
    issues = []
    for col in ['open', 'high', 'low', 'close', 'adj_close', 'volume']:
        rows = df[df[col].isna()]
        # automatic meidum severity if adj_close is missing
        severity = 'medium' if col == 'adj_close' else 'high'
        # add to quality report
        issues.append(make_issues(rows, 'missing_value', f"{col} is missing", severity))
    # concat all six column checks into one dataframe
    return pd.concat(issues, ignore_index= True)

missing_report = detect_missing_values(df)
missing_report["description"].value_counts()

description
adj_close is missing    18374
Name: count, dtype: int64

Dectect non-positive values

In [12]:
PRICE_COLS = ['open', 'high', 'low', 'close', 'adj_close']

# here I flag prices that are negative or +/- inf
def detect_nonpositive_price(df):
    issues = []
    for col in PRICE_COLS:
        x = df[col]
        bad = x.notna() & (~np.isfinite(x) | (x <= 0))
        rows = df[bad]
        desc = col + ' is nonpositive or infinite (' + col + '=' + rows[col].astype(str) + ')'
        issues.append(make_issues(rows, 'nonpositive_price', desc, 'high'))
    return pd.concat(issues, ignore_index= True)

# here I flag impossible OHLC relationships 
def detect_ohlc_violations(df):
    checks = {
        'high < low': df['high'] < df['low'],
        'close > high': df['close'] > df['high'],
        'open > high': df['open'] > df['high'],
        'open < low': df['open'] < df['low'],
        'low > close': df['low'] > df['close']
    }
    issues = []
    for rule, bad in checks.items():
        # catch all rows in df for each OHLC violation
        rows = df[bad]
        desc = (rule + ' (open = ' + rows['open'].astype(str)
                + ', high = ' + rows['high'].astype(str)
                + ', low = ' + rows['low'].astype(str)
                + ', close = ' + rows['close'].astype(str) + ')')
        issues.append(make_issues(rows, 'ohlc_violation', desc, 'high'))
    return pd.concat(issues, ignore_index= True)

# Here I flag zero volume 
def detect_zero_volume(df):
    rows = df[df['volume'] == 0]
    return make_issues(rows, 'zero_volume', 'volume is 0', 'low')

In [13]:
nonpositive_report = detect_nonpositive_price(df)
ohlc_violation_report = detect_ohlc_violations(df)
zero_volume_report = detect_zero_volume(df)

# summary checks and preview of ohlc violations
print("Nonpositive/non-finite prices:", len(nonpositive_report))
print("OHLC violations:", len(ohlc_violation_report))
print(ohlc_violation_report["description"].str.split(" \(").str[0].value_counts())
print("Zero-volume days:", len(zero_volume_report))
ohlc_violation_report.head()

Nonpositive/non-finite prices: 5
OHLC violations: 5
description
open < low    5
Name: count, dtype: int64
Zero-volume days: 75


,ticker,issue_type,issue_date,description,severity
0,PG,ohlc_violation,1962-01-02,"open < low (open = 0.0, high = 1.44, low = 1.4...",high
1,MCD,ohlc_violation,1966-07-05,"open < low (open = 0.0, high = 0.27366, low = ...",high
2,CVX,ohlc_violation,1962-01-02,"open < low (open = 0.0, high = 3.3, low = 3.24...",high
3,JNJ,ohlc_violation,1962-01-02,"open < low (open = 0.0, high = 0.22338, low = ...",high
4,MRK,ohlc_violation,1962-01-02,"open < low (open = 0.0, high = 0.39703, low = ...",high


All 5 nonpositive prices are open = 0.0, and they’re the same 5 rows as the open < low violations. Each falls on the ticker’s first date in the dataset. It's impossible for $0 open since the same day’s low is positive, so this is most likely an unrecorded open stored as 0 by the source. Because these are first observations, there’s no previous price to check. The repair will be handled in 1.3. The 75 zero-volume days are kept and flagged as low severity, since zero volume may reflect non-active or no trading.

In [14]:
# establish clean reference table
with sqlite3.connect(DB_PATH) as conn:
    ref = pd.read_sql("""
    SELECT ticker, ts, close AS ref_close, adj_close as ref_adj_close
    FROM ohlc
    """, conn)
ref.head()

,ticker,ts,ref_close,ref_adj_close
0,JNJ,2024-10-14,161.60,161.60
1,JNJ,2024-10-11,161.46,161.46
2,JNJ,2024-10-10,160.51,160.51
3,JNJ,2024-09-30,162.06,162.06
4,JNJ,2024-09-27,161.40,161.40


In [15]:
# Here I detect return outliers
# ref is used for reference other dectors later down the analysis
def detect_return_outliers(df, ref = None, n_std = 3):
    # Flag abs(return - mean_return) > n_std * std_return within each ticker
    d = df.sort_values(['ticker', 'ts']).copy()

    # compute return = (close - prev_close)/prev_close within each ticker
    prev_close = d.groupby('ticker')['close'].shift(1)
    # exclude undefined prev_close, ensure positive close
    valid_prev = np.isfinite(prev_close) & (prev_close > 0) & np.isfinite(d['close'])
    # calc return
    d['return'] = ((d['close'] - prev_close)/ prev_close).where(valid_prev)

    # calc mean and std
    g = d.groupby('ticker')['return']
    mean_return = g.transform('mean')
    std_return  = g.transform('std')

    # zero or undefined std
    usable_std = std_return.notna() & (std_return > 0)
    skipped = sorted(d.loc[~usable_std, 'ticker'].unique())
    # print explicit message
    if skipped:
        print('Skipped due to 0 or undefined std: ', skipped)

    # Apply flag abs(return - mean_return) > 3 * std_return
    flag = d['return'].notna() & usable_std &((d['return'] - mean_return).abs() > n_std * std_return)
    rows = d[flag]
    n_sd = ((rows['return'] - mean_return[flag]) / std_return[flag]).round(1)
    desc = ('return ' + (rows['return'] * 100).round(2).astype(str) + '% (' + n_sd.astype(str)
            + ' std from ticker mean)')

    # check if flag match the reference
    if ref is not None:
        r = rows[['ticker', 'ts', 'close']].merge(
            ref[['ticker', 'ts', 'ref_close']], 
            on = ['ticker', 'ts'], 
            how = 'left'
        )
        matches = np.isclose(r['close'], r['ref_close'])
        desc = desc + np.where(matches, '; close matches reference', '; close differs from reference')

    return make_issues(rows, 'return_outlier', desc, 'low')
    


In [16]:
# summary view of the outlier report
outlier_report = detect_return_outliers(df, ref)
print('Return outliers: ', len(outlier_report))
print(outlier_report['description'].str.contains('matches reference').value_counts())
outlier_report.head()

Return outliers:  5056
description
True    5056
Name: count, dtype: int64


,ticker,issue_type,issue_date,description,severity
0,AAPL,return_outlier,1980-12-26,return 9.23% (3.3 std from ticker mean); close...,low
1,AAPL,return_outlier,1981-04-02,return 8.77% (3.1 std from ticker mean); close...,low
2,AAPL,return_outlier,1981-09-25,return -12.98% (-4.7 std from ticker mean); cl...,low
3,AAPL,return_outlier,1982-03-19,return 9.02% (3.2 std from ticker mean); close...,low
4,AAPL,return_outlier,1982-11-22,return -8.91% (-3.2 std from ticker mean); clo...,low


In [17]:
# Demo flag spiked return
demo_ret = pd.DataFrame({
    "ticker": ["DEMO"] * 21,
    "ts": pd.date_range("2020-01-01", periods=21).strftime("%Y-%m-%d"),
    "close": [100.0] * 10 + [101.0, 100.0] * 5 + [100.0],
    })
demo_ret.loc[15, "close"] = 200.0
# flag on 2020-01-16
display(detect_return_outliers(demo_ret))

# Demo 0 std is skipped explicitly
flat = pd.DataFrame({"ticker": ["FLAT"] * 3,
                     "ts": ["2020-01-01", "2020-01-02", "2020-01-03"],
                     "close": [5.0, 5.0, 5.0]
                     })
display(detect_return_outliers(flat))

,ticker,issue_type,issue_date,description,severity
0,DEMO,return_outlier,2020-01-16,return 98.02% (3.8 std from ticker mean),low


Skipped due to 0 or undefined std:  ['FLAT']


,ticker,issue_type,issue_date,description,severity


5056 returns (about 1.4% of 368251) exceed 3σ of their ticker’s mean, which is expected because daily returns have fat tails. All 5056 flagged closes match the reference ohlc table, so they aren’t errors introduced into this dataset. 

They’re kept and flagged for review and not deleted. The demos show the detector catching an injected spike and explicitly skipping a ticker with zero std.

Limitation: a single spike inflates its own ticker’s std, which can hide the reversal day.

In [18]:
# detect missing days
def detect_missing_days(df, ref):
    have = df[['ticker', 'ts']].drop_duplicates()
    cal = ref[['ticker', 'ts']].drop_duplicates()
    merged = cal.merge(have, on = ['ticker', 'ts'], how = 'left', indicator = True)
    missing = merged[merged['_merge'] == 'left_only'].sort_values(['ticker', 'ts'])
    return make_issues(missing, 'missing_trade_days', 'date present in ohlc but missing in ohlc_hw'
                       , 'medium')

# summarize "Ticker X is missing Y trading days between [start] and [end]"
def summarize_missing_days(missing_days, df):
    span = df.groupby('ticker')['ts'].agg(start = 'min', end = 'max')
    counts = missing_days.groupby('ticker').size().rename('missing_days')
    s = (span.join(counts)
         .fillna({'missing_days': 0})
         .astype({'missing_days': int})
         .reset_index()
         )
    s['summary'] = ('Ticker ' + s['ticker'] + ' is missing ' + s['missing_days'].astype(str)
                    + ' trading days between '+ s['start'] + ' and ' + s['end'])
    return s

In [19]:
missing_day_report = detect_missing_days(df, ref)
print('Missing trading days: ', len(missing_day_report))
for line in summarize_missing_days(missing_day_report, df)['summary']:
    print(line)

Missing trading days:  0
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-0

In [20]:
# Demo to remove one trading days and check detector functionality
test_ref = pd.DataFrame({'ticker': ['AAPPL'] * 3,
                         'ts': ['2020-01-02', '2020-01-03', '2020-01-06']})
# drop one record in the test
test_df = test_ref.drop(index = 1)
demo_missing = detect_missing_days(test_df, test_ref)
display(demo_missing)
print(summarize_missing_days(demo_missing, test_ref)['summary'].iloc[0])

,ticker,issue_type,issue_date,description,severity
0,AAPPL,missing_trade_days,2020-01-03,date present in ohlc but missing in ohlc_hw,medium


Ticker AAPPL is missing 1 trading days between 2020-01-02 and 2020-01-06


Reference calendar is each ticker’s dates in the supplied ohlc table, over that ticker’s own history. Business weekdays alone would wrongly flag exchange holidays, and a common calendar would wrongly flag dates before a ticker’s first listing. 

Result: 0 missing trading days for all 30 tickers. The demo confirms the detector catches a intentionally removed date.

In [21]:
# detect adjusted close price inconsistency
def detect_adj_close_inconsistency(df, ref):
    m = df.merge(ref[['ticker', 'ts', 'ref_close', 'ref_adj_close']],
                 on = ['ticker', 'ts'], 
                 how='left')
    factor = m['adj_close'] / m['close']
    ref_factor = m['ref_adj_close'] / m['ref_close']
    bad = (m['adj_close'].notna() & ref_factor.notna() & ~np.isclose(factor, ref_factor))
    rows = m[bad]
    ratio = (factor[bad] / ref_factor[bad]).round(2).astype(str)
    desc = 'adj_close inconsistent with ref adj factor (' + ratio + 'x expected)'
    return make_issues(rows, 'adj_close_inconsistency', desc, 'high')

In [22]:
adj_close_report = detect_adj_close_inconsistency(df, ref)
print('adj_close inconsistency: ', len(adj_close_report))
adj_close_report.head()

adj_close inconsistency:  10417


,ticker,issue_type,issue_date,description,severity
0,CSCO,adj_close_inconsistency,1992-07-30,adj_close inconsistent with ref adj factor (11...,high
1,BA,adj_close_inconsistency,2013-08-26,adj_close inconsistent with ref adj factor (17...,high
2,HON,adj_close_inconsistency,2021-12-29,adj_close inconsistent with ref adj factor (5....,high
3,TRV,adj_close_inconsistency,1987-02-18,adj_close inconsistent with ref adj factor (23...,high
4,HD,adj_close_inconsistency,1991-08-21,adj_close inconsistent with ref adj factor (11...,high


10417 adj_close values are inflated 4–24× larger compared to the reference adjustment factor (adj_close / close), while open, high, low, close and volume all match the reference. 

Together with the 18374 missing adj_close values, these will be addressed in 1.3.

In [23]:
# detect duplicates
def detect_duplicates(dup_rows):
    return make_issues(dup_rows, 'exact_duplicates', 'exact duplicates record (removed); original ' \
    'label: ' + dup_rows['ticker_raw'], 'medium')

# detect conflicting keys 
def detect_conflicting_keys(df):
    counts = df.groupby(['ticker', 'ts']).size()
    keys = counts[counts > 1].reset_index()[['ticker', 'ts']]
    return make_issues(keys, 'conflicting_keys', 'multiple rows with different values for the same' \
    '(ticker, ts)', 'high')

duplicate_report = detect_duplicates(duplicate_rows)
conflict_report = detect_conflicting_keys(df)
print('Exact duplicates:', len(duplicate_report), '; Conflicting keys:', len(conflict_report)) 

Exact duplicates: 29460 ; Conflicting keys: 0


In [24]:
# combine all data detection reports
quality_report = pd.concat([duplicate_report, conflict_report, 
                            missing_day_report, nonpositive_report, 
                            ohlc_violation_report, zero_volume_report, 
                            outlier_report, missing_report, adj_close_report]
                            , ignore_index=True)
quality_report.to_csv("data_quality_report_before.csv", index=False)
print('Total issues: ', len(quality_report))
quality_report.groupby(['issue_type', 'severity']).size()

Total issues:  63392


issue_type               severity
adj_close_inconsistency  high        10417
exact_duplicates         medium      29460
missing_value            medium      18374
nonpositive_price        high            5
ohlc_violation           high            5
return_outlier           low          5056
zero_volume              low            75
dtype: int64

In [25]:
# Injected cases absent from ohlc_hw
demo = pd.DataFrame({
    "ticker": ["DEMO"] * 6,
    "ts": ["2020-01-02", "2020-01-03", "2020-01-06", "2020-01-07", "2020-01-08", "2020-01-08"],
    "open": [10.0, 10.0, 10.0, 10.0, 10.0, 10.0],
    "high": [11.0,  9.8, 11.0,  9.0, 11.0, 11.0],
    "low": [ 9.0,  9.5, 10.5, 10.0,  9.0,  9.0],
    "close": [10.5, 10.2, 10.2,  9.5, np.nan, 10.4],
    "adj_close": [10.5, 10.2, 10.2,  9.5, np.inf, 10.4],
    "volume": [100, 100, 100, 100, 100, 100],
})
# row 0: clean 
# row 1: close > high, open > high 
# row 2: low > close, open < low
# row 3: high < low, close > high, open > high, low > close
# row 4: missing close + infinite adj_close,  rows 4-5: conflicting (ticker, ts)
(pd.concat([detect_missing_values(demo), detect_nonpositive_price(demo),
           detect_ohlc_violations(demo), detect_conflicting_keys(demo)], ignore_index=True)
           .sort_values(["issue_date", "issue_type"])
           .reset_index(drop=True))

,ticker,issue_type,issue_date,description,severity
0,DEMO,ohlc_violation,2020-01-03,"close > high (open = 10.0, high = 9.8, low = 9...",high
1,DEMO,ohlc_violation,2020-01-03,"open > high (open = 10.0, high = 9.8, low = 9....",high
2,DEMO,ohlc_violation,2020-01-06,"open < low (open = 10.0, high = 11.0, low = 10...",high
3,DEMO,ohlc_violation,2020-01-06,"low > close (open = 10.0, high = 11.0, low = 1...",high
4,DEMO,ohlc_violation,2020-01-07,"high < low (open = 10.0, high = 9.0, low = 10....",high
5,DEMO,ohlc_violation,2020-01-07,"close > high (open = 10.0, high = 9.0, low = 1...",high
6,DEMO,ohlc_violation,2020-01-07,"open > high (open = 10.0, high = 9.0, low = 10...",high
7,DEMO,ohlc_violation,2020-01-07,"low > close (open = 10.0, high = 9.0, low = 10...",high
8,DEMO,conflicting_keys,2020-01-08,multiple rows with different values for the sa...,high
9,DEMO,missing_value,2020-01-08,close is missing,high


This is the full diagnosis before any repairs with total of 63392 issue rows. These are saved to data_quality_report_before.csv. All detectors will be rerun on the cleaned data in 1.3 to confirm the repairs.

### 1.3 Clean the Data

Repair choices (each repaired value keeps a flag column):
- **Missing raw prices:** forward-fill within each ticker (sorted by date); leading gaps are quarentined.
- **Missing volume:** filled with 0 and flagged (unknown volume is not evidence of no trading).
- **Rows with all raw prices missing:** dropped before any filling.
- **Impossible open** (nonpositive or outside [low, high]): replaced with the same day's close, between [low, high] (Option 3, using the other prices)
- **Impossible close** (nonpositive or outside [low, high], including after forward-fill): clipped into [low, high].
- **adj_close** (missing, non-finite, inconsistent with the reference factor, or its close was repaired): rebuilt as `close × (ref_adj_close / ref_close)` for the same (ticker, ts) from the supplied `ohlc` table. Factor not forward-filled.
- **Return outliers and zero-volume days:** kept unchanged and flagged for review (all outlier closes match the reference).
- Rows that still fail validation are moved to a quarantine table with a reason.

In [26]:
# all repaires
FLAG_COLS = ["flag_price_ffill", "flag_volume_filled", 
             "flag_open_repaired", "flag_close_repaired",
             "flag_adj_close_rebuilt", "flag_return_outlier", "flag_zero_volume"]
# targets
RAW_PRICES = ["open", "high", "low", "close"]

# create initial flag for all rows
def init_flags(df):
    out = df.sort_values(['ticker', 'ts']).reset_index(drop=True).copy()
    for c in FLAG_COLS:
        out[c] = False
    out['adj_close_source'] = 'vendor'
    return out

# drop rows where all raw prices are missing before filling
def drop_all_price_missing(df):
    all_missing = df[RAW_PRICES].isna().all(axis=1)
    removed = df[all_missing].assign(reason='all raw prices are missing')
    return df[~all_missing].copy(), removed

# foward-fill each ticker's raw price. leading gaps stay NaN and are quarantined later
def ffill_missing_price(df):
    out = df.sort_values(['ticker', 'ts']).copy()
    for col in RAW_PRICES:
        was_missing = out[col].isna()
        out[col] = out.groupby('ticker')[col].ffill()
        # flag rows where a missing value was actually filled
        # |= keeps flags from earlier columns
        out["flag_price_ffill"] |= was_missing & out[col].notna()
    return out 

# unknow volume as 0 with a flag (0 as unknown not necessarily no trading)
def fill_missing_volume(df):
    out = df.copy()
    missing = out['volume'].isna()
    # fill only the missing volumes with 0 (flagged next as unknown)
    out.loc[missing, 'volume'] = 0
    out["flag_volume_filled"] |= missing
    return out

# all impossible ohlc values
def repair_ohlc(df):
    out = df.copy()
    l, h = out['low'], out['high']
    valid_range = (l >0) & (h >= l)

    # bad close scenarios
    bad_close = valid_range & ((out["close"] <= 0) | (out["close"] < l) | (out["close"] > h))
    # move each invalid close to the nearest bound of its own day's [low, high]
    out.loc[bad_close, "close"] = out.loc[bad_close, "close"].clip(lower=l[bad_close], upper=h[bad_close])
    out["flag_close_repaired"] |= bad_close

    # bad open scenarios
    bad_open = valid_range & ((out["open"] <= 0) | (out["open"] < l) | (out["open"] > h))
    # move each invalid open to the nearest bound of its own day's [low, high]
    out.loc[bad_open, "open"] = out.loc[bad_open, "close"].clip(lower=l[bad_open], upper=h[bad_open])
    out["flag_open_repaired"] |= bad_open
    return out

# rebuild adj_close
# adj_close = close x (ref_adj_close / ref_close) for the same (ticker, ts) when the
# vendor value is missing, non-finite, inconsistent with the reference factor, or its
# close was repaired. No forward-filling
def rebuild_adj_close(df, ref_df):
    out = df.merge(ref_df[['ticker', 'ts', 'ref_close', 'ref_adj_close']], on=['ticker', 'ts'], how='left')
    # calculate factors
    ref_factor = out['ref_adj_close']/out['ref_close']
    vendor_factor = out['adj_close']/out['close']
    needs = (~np.isfinite(out["adj_close"]) | (out["adj_close"] <= 0)
             | ~np.isclose(vendor_factor, ref_factor)
             | out["flag_close_repaired"])
    # check fixable conditions
    fixable = needs & np.isfinite(ref_factor) & (ref_factor > 0)

    # adj_close = close x (ref_adj_close / ref_close)
    out.loc[fixable, "adj_close"] = out.loc[fixable, "close"] * ref_factor[fixable]
    out.loc[fixable, "adj_close_source"] = "close x ohlc reference factor"
    out["flag_adj_close_rebuilt"] |= fixable
    # qurantine unresolved ones
    out.loc[needs & ~fixable, "adj_close"] = np.nan

    return out.drop(columns=["ref_close", "ref_adj_close"])

# flag issues
def add_review_flags(df, outlier_report, zero_volume_report):
    out = df.copy()
    keys = list(zip(out["ticker"], out["ts"]))
    outlier_keys = set(zip(outlier_report["ticker"], outlier_report["issue_date"]))
    zero_vol_keys = set(zip(zero_volume_report["ticker"], zero_volume_report["issue_date"]))
    out["flag_return_outlier"] = [k in outlier_keys for k in keys]
    out["flag_zero_volume"] = [k in zero_vol_keys for k in keys]
    return out

# move rows that still fail to quarantine with reasons
def quarantine_invalid(df):
    prices = df[RAW_PRICES + ['adj_close']]
    reasons = pd.Series('', index = df.index)
    # assign reasons
    reasons[prices.isna().any(axis=1)] += "unresolved missing price; "
    reasons[(~np.isfinite(prices.fillna(1)) | (prices.fillna(1) <= 0)).any(axis=1)] += "nonpositive/ non-finite price; "
    reasons[(df["high"] < df["low"]) | (df["open"] < df["low"]) | (df["open"] > df["high"])
            | (df["close"] < df["low"]) | (df["close"] > df["high"])] += "OHLC bounds violation; "
    bad = reasons != ''
    # return correct data and format reason string
    return df[~bad].copy(), df[bad].assign(reason=reasons[bad].str.rstrip("; "))


In [27]:
# clean ohlc pipeline
def clean_ohlc(df, ref, outlier_report, zero_volume_report):
    # run the data through the cleaning pipeline built
    d = init_flags(df)
    d, removed = drop_all_price_missing(d)
    d = ffill_missing_price(d)
    d = fill_missing_volume(d)
    d = repair_ohlc(d)
    d = rebuild_adj_close(d, ref )
    d = add_review_flags(d, outlier_report, zero_volume_report)
    clean, quarantined = quarantine_invalid(d)
    return clean, removed, quarantined

# summary view
clean, removed, quarantined = clean_ohlc(df, ref, outlier_report, zero_volume_report)
print("Rows in:", len(df), "| retained:", len(clean),
      "| removed:", len(removed), "| quarantined:", len(quarantined))
print(clean[FLAG_COLS].sum())

Rows in: 368251 | retained: 368251 | removed: 0 | quarantined: 0
flag_price_ffill              0
flag_volume_filled            0
flag_open_repaired            5
flag_close_repaired           0
flag_adj_close_rebuilt    28791
flag_return_outlier        5056
flag_zero_volume             75
dtype: int64


In [28]:
# Demos with injected cases where the original data is missing to test cleaning functionality
demo = pd.DataFrame({
    "ticker": ["DEMO"] * 6 + ["LEAD"] * 2,
    "ts": ["2020-01-02", "2020-01-03", "2020-01-06", "2020-01-07", "2020-01-08", "2020-01-09",
           "2020-01-02", "2020-01-03"],
    "open":  [10.0, np.nan, 10.0,  0.0, np.nan, 10.0, np.nan, 5.0],
    "high":  [11.0, np.nan, 10.5, 11.0, np.nan, 11.0, np.nan, 5.5],
    "low":   [ 9.0, np.nan,  9.8,  9.0, np.nan,  9.0, np.nan, 4.5],
    "close": [10.5, 10.6,  np.nan, 10.0, np.nan, 12.0,  5.2,  5.0],
    "adj_close": [10.5, np.nan, np.nan, 10.0, np.nan, 12.0, 5.2, 5.0],
    "volume": [100, np.nan, 100, 100, np.nan, 100, 100, 100],
})

# Demo cases and expect behaviors:
# 01-03: missing o/h/l + volume -> forward-filled, volume 0 -> flag, adj_close rebuilt
# 01-06: missing close -> ffill gives 10.6 > high 10.5 -> clipped to 10.5
# 01-07: open = 0 -> replaced with same day close
# 01-08: all raw prices missing -> removed before filling
# 01-09: close 12 > high 11 -> clipped to 11
# LEAD 01-02: nothing earlier to fill from -> quarantined

demo_ref = demo[["ticker", "ts"]].assign(ref_close=1.0, ref_adj_close=1.0)
no_flags = pd.DataFrame(columns=["ticker", "issue_date"])
d_clean, d_removed, d_quaran = clean_ohlc(demo, demo_ref, no_flags, no_flags)
# after cleaning
display(d_clean[["ticker", "ts"] + RAW_PRICES + ["adj_close", "volume"] + FLAG_COLS[:5]])
# removal ones
display(d_removed[["ticker", "ts", "reason"]])
# quarantine ones
display(d_quaran[["ticker", "ts", "reason"]])

,ticker,ts,open,high,low,close,adj_close,volume,flag_price_ffill,flag_volume_filled,flag_open_repaired,flag_close_repaired,flag_adj_close_rebuilt
0,DEMO,2020-01-02,10.0,11.0,9.0,10.5,10.5,100.0,False,False,False,False,False
1,DEMO,2020-01-03,10.0,11.0,9.0,10.6,10.6,0.0,True,True,False,False,True
2,DEMO,2020-01-06,10.0,10.5,9.8,10.5,10.5,100.0,True,False,False,True,True
3,DEMO,2020-01-07,10.0,11.0,9.0,10.0,10.0,100.0,False,False,True,False,False
4,DEMO,2020-01-09,10.0,11.0,9.0,11.0,11.0,100.0,False,False,False,True,True
6,LEAD,2020-01-03,5.0,5.5,4.5,5.0,5.0,100.0,False,False,False,False,False


,ticker,ts,reason
4,DEMO,2020-01-08,all raw prices are missing


,ticker,ts,reason
5,LEAD,2020-01-02,unresolved missing price


In [29]:
# Validations before/ after
def validate(df):
    p = df[RAW_PRICES + ["adj_close"]]
    return pd.Series({
        # one sum across all rows
        "missing prices": int(p.isna().sum().sum()),
        "non-finite prices": int((~np.isfinite(p)).sum().sum()),
        "nonpositive prices": int((p <= 0).sum().sum()),
        "duplicate (ticker, ts) keys": int(df.duplicated(["ticker", "ts"]).sum()),
        "high < low": int((df["high"] < df["low"]).sum()),
        "open outside [low, high]": int(((df["open"] < df["low"]) | (df["open"] > df["high"])).sum()),
        "close outside [low, high]": int(((df["close"] < df["low"]) | (df["close"] > df["high"])).sum()),
        "missing/non-finite/negative volume": int((~np.isfinite(df["volume"]) | (df["volume"] < 0)).sum()),
        "adj_close inconsistent with reference": len(detect_adj_close_inconsistency(df, ref)),
        # ISO dates (YYYY-MM-DD)
        "non-ISO ts": int((~df["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")).sum()),
    })

display(pd.DataFrame({"before": validate(df), "after": validate(clean)}))

stats_cols = ["open", "close", "adj_close", "volume"]
# summary view before/after
pd.concat({"before": df[stats_cols].describe().T[["count", "min", "max", "mean"]],
           "after": clean[stats_cols].describe().T[["count", "min", "max", "mean"]]}, axis=1).round(3)

,before,after
missing prices,18374,0
non-finite prices,18374,0
nonpositive prices,5,0
"duplicate (ticker, ts) keys",0,0
high < low,0,0
"open outside [low, high]",5,0
"close outside [low, high]",0,0
missing/non-finite/negative volume,0,0
adj_close inconsistent with reference,10417,0
non-ISO ts,0,0


before                                        after         \
              count    min           max          mean     count    min   
open       368251.0  0.000  6.042600e+02  4.630900e+01  368251.0  0.005   
close      368251.0  0.005  6.054000e+02  4.631700e+01  368251.0  0.005   
adj_close  349877.0  0.003  1.044251e+04  5.171200e+01  368251.0  0.003   
volume     368251.0  0.000  9.276606e+09  3.095142e+07  368251.0  0.000   

                                       
                    max          mean  
open       6.042600e+02  4.630900e+01  
close      6.054000e+02  4.631700e+01  
adj_close  6.054000e+02  3.727400e+01  
volume     9.276606e+09  3.095142e+07

In [30]:
# save cleaned data and display counts
with sqlite3.connect(DB_PATH) as conn:
    clean.to_sql('ohlc_hw_cleaned', conn, if_exists= 'replace', index=False)
    print(pd.read_sql("SELECT COUNT(*) AS rows, MIN(ts) AS first, MAX(ts) AS last " \
    "FROM ohlc_hw_cleaned", conn))
    print("Raw ohlc_hw rows (unchanged):", 
          pd.read_sql("SELECT COUNT(*) AS n FROM ohlc_hw", conn)["n"][0])

     rows       first        last
0  368251  1962-01-02  2024-12-06
Raw ohlc_hw rows (unchanged): 397711


In [31]:
# show cleaning results after each step of the pipeline
print(f"Raw rows: {len(raw)} \n-> exact duplicates removed: {len(duplicate_rows)} "
      f"\n-> cleaned: {len(df)} \n-> removed: {len(removed)}, quarantined: {len(quarantined)} "
      f"\n-> saved: {len(clean)}")

Raw rows: 397711 
-> exact duplicates removed: 29460 
-> cleaned: 368251 
-> removed: 0, quarantined: 0 
-> saved: 368251
